# 16S Analyses of the Longitudinal Acne Study
## V1-V3 and V4 primer sets comparison

Date created: 12/28/2024

Notebook authors: Yang Chen

Data analysis by: Yang Chen, Tyler Myers, Britta De Pessemier

This notebook plots the following:

- Plot showing abundance of Cutibacterium in each sample with each primer pair (i.e. the axes are V13 vs V4, each point is the amount of Cutibacterium in one sample by each of the primer pairs)
- Venn diagram illustrating the overlap of genera-level taxa detected by both primer sets, alongside those unique to V1-V3 or V4

In [879]:
# Import Python packages
import pandas as pd
import numpy as np
import biom
import matplotlib.pyplot as plt
import seaborn as sns
import gemelli
from gemelli.preprocessing import rclr_transformation
from matplotlib_venn import venn2, venn2_circles, venn3
from venn import venn
from Bio import SeqIO
from matplotlib.patches import Circle
from scipy.stats import pearsonr
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import pearsonr, norm
from sklearn.decomposition import PCA
from biom import Table
import os
from upsetplot import from_memberships, UpSet
from skbio.stats.composition import clr
from skbio.stats.distance import mantel, DistanceMatrix
from scipy.spatial import procrustes
from scipy.spatial.distance import pdist, squareform



In [880]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    return df


In [881]:
# Load tables
V1V3_biom = biom_to_df('../Data/16S/Tables/16S_V1-V3_Genus_collapsed.biom')
V4_biom = biom_to_df('../Data/16S/Tables/16S_V4_Genus_collapsed.biom')
V1V3_biom.index = V1V3_biom.index.str.replace('^ g__', '', regex=True)
V4_biom.index = V4_biom.index.str.replace('^ g__', '', regex=True)
V1V3_biom

,LAMI.RD001.D0.C1,LAMI.RD001.D0.C2,LAMI.RD001.D14.C1,LAMI.RD001.D14.C2,LAMI.RD001.D28.C1,LAMI.RD002.D0.C2,LAMI.RD003.D14.C1,LAMI.RD002.D14.C1,LAMI.RD003.D28.C1,LAMI.RD001.D28.C2,...,LAMI.RD319.D28.C3,LAMI.RD319.D11.C1,LAMI.RD319.D21.C2,LAMI.RD319.D7.C3,LAMI.RD318.D28.C3,LAMI.RD319.D14.C1,LAMI.RD319.D21.C3,LAMI.RD319.D14.C2,LAMI.RD319.D9.C1,LAMI.RD319.D9.C2
Cutibacterium,0.626691,0.479703,0.592730,0.467233,0.628018,0.769435,0.874503,0.912709,0.628018,0.607058,...,0.447334,0.346246,0.692757,0.260281,0.853807,0.504112,0.266118,0.614221,0.326347,0.613956
uncultured,0.000000,0.000531,0.000531,0.000000,0.002919,0.000000,0.013797,0.001327,0.033696,0.000000,...,0.537543,0.642080,0.297957,0.732024,0.142743,0.489785,0.727249,0.374105,0.660918,0.374370
Staphylococcus,0.024940,0.041921,0.022022,0.036349,0.031308,0.098965,0.025206,0.027859,0.027328,0.054126,...,0.002653,0.000000,0.002653,0.005572,0.001327,0.002123,0.002919,0.002123,0.003449,0.002123
Streptococcus,0.077474,0.099231,0.063677,0.118334,0.077739,0.042186,0.003715,0.011409,0.144070,0.044839,...,0.000000,0.002123,0.000000,0.000000,0.000000,0.000000,0.000265,0.000000,0.000265,0.000000
Corynebacterium,0.046431,0.093393,0.130539,0.120987,0.096843,0.010348,0.036614,0.013001,0.008225,0.105864,...,0.002653,0.003715,0.001857,0.000531,0.000000,0.001592,0.001061,0.002388,0.000796,0.003715
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Methylotenera,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Fusicatenibacter,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Ruminococcus,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Lachnospira,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [882]:
# Convert indices to sets for set operations
indices_V1V3 = set(V1V3_biom.index)
indices_V4 = set(V4_biom.index)

# Taxa unique to V1V3
unique_to_V1V3 = indices_V1V3 - indices_V4

# Taxa unique to V4
unique_to_V4 = indices_V4 - indices_V1V3

# Taxa shared by both
shared_taxa = indices_V1V3 & indices_V4

# Step 2: Filter shared and unique taxa based on 10% sample prevalence
def filter_by_prevalence(df, taxa, prevalence_threshold=0.1):
    # Subset the DataFrame to include only the specified taxa
    subset_df = df.loc[list(taxa)]  # Convert set to list for indexing
    # Calculate prevalence: proportion of samples where the taxon is present
    prevalence = (subset_df > 0).sum(axis=1) / subset_df.shape[1]
    # Filter taxa based on the prevalence threshold
    return set(prevalence[prevalence >= prevalence_threshold].index)

# Apply prevalence filtering
filtered_shared = filter_by_prevalence(V1V3_biom, shared_taxa, prevalence_threshold=0.1)
filtered_unique_to_V1V3 = filter_by_prevalence(V1V3_biom, unique_to_V1V3, prevalence_threshold=0.1)
filtered_unique_to_V4 = filter_by_prevalence(V4_biom, unique_to_V4, prevalence_threshold=0.1)

# Step 3: Create the Venn diagram with enhancements
plt.clf()
plt.figure(figsize=(8, 8))  # Increased figure size for better clarity

# Create the Venn diagram with customized circle outlines
venn = venn2(
    [filtered_unique_to_V1V3 | filtered_shared, filtered_unique_to_V4 | filtered_shared],
    ('', ''),
    set_colors=('lightblue', 'lightgreen'),  # Fill colors for the circles
    alpha=0.4  # Transparency for fill colors
)

# Customize the circle outlines with darker colors
for circle, color in zip(['10', '01'], ['blue', 'green']):
    venn.get_patch_by_id(circle).set_edgecolor(color)  # Darker outline color
    venn.get_patch_by_id(circle).set_linewidth(2)  # Thickness of the outline

# Customizing colors for the Venn diagram (switching green and purple)
venn.get_patch_by_id('10').set_color('#87CEEB')  # Light blue for V1-V3
venn.get_patch_by_id('01').set_color('#DDA0DD')  # Light purple for V4
venn.get_patch_by_id('11').set_color('#98FB98')  # Light green for shared

# Customizing text labels
for id in ['10', '01', '11']:
    if venn.get_label_by_id(id):
        venn.get_label_by_id(id).set_fontsize(12)  # Larger font size
        venn.get_label_by_id(id).set_color('black')  # Black text for better readability

# Add a title with larger font size and weight
plt.title('Bacterial Genera Resolved by 16S V1-V3 vs V4',
          fontsize=18)

# Add a legend for the groups
plt.legend(
    handles=[
        plt.Line2D([0], [0], marker='o', color='#87CEEB', lw=0, label='V1-V3 Unique'),
        plt.Line2D([0], [0], marker='o', color='#98FB98', lw=0, label='Shared'),
        plt.Line2D([0], [0], marker='o', color='#DDA0DD', lw=0, label='V4 Unique'),
    ],
    loc='lower center', bbox_to_anchor=(0.5, -0.075), ncol=3, fontsize=12
)

# Save the figure with a higher resolution
plt.savefig('../Figures/16S_Figures/primer_comparison/primer_venn_diagram.png', dpi=600, bbox_inches='tight')

# Show the figure (optional)
plt.show()

# Print the results
print("Filtered Unique to V1V3 (>=10% prevalence):")
print(filtered_unique_to_V1V3)

print("\nFiltered Unique to V4 (>=10% prevalence):")
print(filtered_unique_to_V4)

print("\nFiltered Shared Taxa (>=10% prevalence):")
print(filtered_shared)


Filtered Unique to V1V3 (>=10% prevalence):
{'Reyranella', 'Mycobacterium', 'Janibacter'}

Filtered Unique to V4 (>=10% prevalence):
{'Bifidobacterium', 'Leptotrichia', 'Turicella', 'Chryseobacterium', 'Alloiococcus', 'Fenollaria', 'Empedobacter', 'Stenotrophomonas', 'Massilia', 'Geobacillus', 'Moraxella', 'Abiotrophia', 'Blautia', 'Lactococcus', 'Brachybacterium', 'Peptoniphilus', 'Leuconostoc', 'Vibrio', 'Psychrobacter', 'Peptostreptococcus', 'Fusobacterium', 'Aerococcus', 'Capnocytophaga', 'Dolosigranulum', 'Gardnerella', 'Jeotgalicoccus', 'Luteimonas', 'Aeromonas', 'Frederiksenia', 'Atopobium', 'Pantoea', 'Enhydrobacter', 'Finegoldia', 'Bradyrhizobium', 'Marinomonas', 'Aggregatibacter'}

Filtered Shared Taxa (>=10% prevalence):
{'Limnobacter', 'uncultured', 'Streptococcus', 'Brevundimonas', 'Caulobacter', 'Veillonella', 'Actinomyces', 'Gemella', 'Porphyromonas', 'Cutibacterium', 'Anaerococcus', 'Phenylobacterium', 'Nocardioides', 'Corynebacterium', 'Sphingopyxis', 'Alloprevotella',

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_59393/4290181383.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Create three-way venn diagram of 16S V1-V3, 16S V4, and metaG from rs210 OR wolr2

In [883]:
# import metaG rs210 or wolr2 table
# db = 'rs210'
db = 'wolr2'
metaG_db_path = f'../Data/metaG/Tables/metaG_{db}_micov-filt_genus.biom'
metaG_db_df = biom_to_df(metaG_db_path)
metaG_db_df.index = metaG_db_df.index.str.replace('^ g__', '', regex=True)

metaG_db_df

,LAMI.RD306.D14.C1,LAMI.RD306.D0.C3,LAMI.RD310.D16.C2,LAMI.RD310.D0.C2,LAMI.RD310.D14.C3,LAMI.RD310.D7.C3,LAMI.RD310.D0.C3,LAMI.RD310.D21.C2,LAMI.RD304.D11.C1,LAMI.RD304.D0.C1,...,LAMI.RD308.D0.C2,LAMI.RD308.D0.C3,LAMI.RD308.D23.C2,LAMI.RD308.D14.C3,LAMI.RD308.D11.C2,LAMI.RD308.D9.C3,LAMI.RD306.D11.C1,LAMI.RD306.D23.C1,LAMI.RD306.D14.C3,LAMI.RD306.D28.C3
Cutibacterium,0.885295,0.957772,0.984448,0.978453,0.994953,0.995983,0.989133,0.980080,0.969096,0.964177,...,0.984800,0.994262,0.874065,0.994297,0.985900,0.969424,0.910988,0.984765,0.936587,0.924127
Corynebacterium,0.026314,0.007272,0.003572,0.009076,0.002600,0.001534,0.006265,0.010985,0.004157,0.008947,...,0.002225,0.001651,0.011863,0.001780,0.003103,0.003935,0.014217,0.007951,0.015095,0.020423
Actinomyces,0.026079,0.007483,0.000398,0.000726,0.000117,0.000164,0.000445,0.002529,0.003736,0.001358,...,0.002330,0.000913,0.015645,0.000831,0.001429,0.008502,0.010645,0.001745,0.010434,0.014310
Micrococcus,0.005293,0.005844,0.001077,0.006148,0.000281,0.000176,0.001604,0.000972,0.003197,0.005855,...,0.001639,0.001230,0.002857,0.000773,0.002541,0.004836,0.006874,0.000539,0.003068,0.010879
Staphylococcus,0.005644,0.002459,0.004836,0.000574,0.001066,0.001370,0.000386,0.001335,0.004333,0.006979,...,0.000457,0.000504,0.000539,0.000082,0.000480,0.000504,0.004239,0.001733,0.005890,0.005808
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Mogibacterium,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000082,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Nanogingivalis,0.000047,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000023,0.000000,0.000000,0.000000,0.000082,0.000000,0.000000,0.000000
Scardovia,0.000012,0.000000,0.000023,0.000000,0.000000,0.000000,0.000000,0.000000,0.000012,0.000000,...,0.000000,0.000000,0.000082,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Latilactobacillus,0.000035,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000012,0.000000,0.000000,0.000000,0.000047,0.000000,0.000035,0.000000


In [884]:
# Step 1: Convert indices to sets
indices_V1V3 = set(V1V3_biom.index)
indices_V4 = set(V4_biom.index)
indices_metaG = set(metaG_db_df.index)

# Step 2: Filter by prevalence (>=10% of samples)
filtered_V1V3 = filter_by_prevalence(V1V3_biom, indices_V1V3, prevalence_threshold=0.1)
filtered_V4 = filter_by_prevalence(V4_biom, indices_V4, prevalence_threshold=0.1)
filtered_metaG = filter_by_prevalence(metaG_db_df, indices_metaG, prevalence_threshold=0.1)

# Step 3: Create Venn diagram
plt.clf()
plt.figure(figsize=(10, 10))

venn = venn3(
    [filtered_V1V3, filtered_V4, filtered_metaG],
    set_labels=('V1–V3', 'V4', f'metaG {db}'),
    set_colors=('#87CEEB', '#DDA0DD', '#90EE90'),  # Blue, Purple, Light Green
    alpha=0.4,
)
# Change font size of set labels
for text in venn.set_labels:
    text.set_fontsize(16)  # Set to desired size


# Customize labels
for id in ['100', '010', '001', '110', '101', '011', '111']:
    label = venn.get_label_by_id(id)
    if label:
        label.set_fontsize(14)
        label.set_color('black')

# Title
plt.title(f'Concordant Genera from Microbiome Data Types',
          fontsize=20)

# Legend (optional, since labels are present)
# plt.legend(
#     handles=[
#         plt.Line2D([0], [0], marker='o', color='#87CEEB', lw=0, label='16S V1–V3'),
#         plt.Line2D([0], [0], marker='o', color='#DDA0DD', lw=0, label='16S V4'),
#         plt.Line2D([0], [0], marker='o', color='#90EE90', lw=0, label=f'metaG {db}'),
#     ],
#     loc='upper left',  # or use 'best' / 'center left' / 'outside' if you want it external
#     ncol=1,
#     fontsize=14
# )

# Save and show
plt.savefig(f'../Figures/16S_Figures/primer_comparison/primer_vs_metaG-{db}_venn.png', dpi=600, bbox_inches='tight')
plt.show()

# Print results
print("Filtered V1V3 Genera (>=10% prevalence):", len(filtered_V1V3))
print("Filtered V1V3 Genera (>=10% prevalence):", filtered_V1V3)

print("Filtered V4 Genera (>=10% prevalence):", len(filtered_V4))
print("Filtered Shotgun Genera (>=10% prevalence):", len(filtered_metaG))


Filtered V1V3 Genera (>=10% prevalence): 33
Filtered V1V3 Genera (>=10% prevalence): {'Limnobacter', 'uncultured', 'Streptococcus', 'Brevundimonas', 'Caulobacter', 'Veillonella', 'Actinomyces', 'Gemella', 'Porphyromonas', 'Cutibacterium', 'Anaerococcus', 'Phenylobacterium', 'Nocardioides', 'Corynebacterium', 'Sphingopyxis', 'Alloprevotella', 'Lactobacillus', 'Pseudomonas', 'Thermus', 'Paracoccus', 'Haemophilus', 'Micrococcus', 'Brevibacterium', 'Neisseria', 'Mycobacterium', 'Lysobacter', 'Lawsonella', 'Reyranella', 'Staphylococcus', 'Janibacter', 'Rothia', 'Prevotella', 'Kocuria'}
Filtered V4 Genera (>=10% prevalence): 74
Filtered Shotgun Genera (>=10% prevalence): 111


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_59393/3393666175.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [885]:
# Dictionary to store taxa in each Venn region
venn_regions = {
    'V1V3_only': filtered_V1V3 - filtered_V4 - filtered_metaG,
    'V4_only': filtered_V4 - filtered_V1V3 - filtered_metaG,
    'metaG_only': filtered_metaG - filtered_V1V3 - filtered_V4,
    'V1V3_and_V4': (filtered_V1V3 & filtered_V4) - filtered_metaG,
    'V1V3_and_metaG': (filtered_V1V3 & filtered_metaG) - filtered_V4,
    'V4_and_metaG': (filtered_V4 & filtered_metaG) - filtered_V1V3,
    'all_three': filtered_V1V3 & filtered_V4 & filtered_metaG
}

# Optional: Print results
for region, taxa in venn_regions.items():
    print(f"\n{region} ({len(taxa)} taxa):")
    for t in sorted(taxa):
        print(f"  - {t}")

# Optional: Save to CSV
import pandas as pd
output_dir = '../Figures/16S_Figures/primer_comparison/venn_taxa_lists/'
os.makedirs(output_dir, exist_ok=True)

for region, taxa in venn_regions.items():
    pd.Series(sorted(taxa)).to_csv(f"{output_dir}{region}_{db}.csv", index=False, header=['Taxon'])



V1V3_only (1 taxa):
  - Reyranella

V4_only (27 taxa):
  - Aeromonas
  - Allorhizobium-Neorhizobium-Pararhizobium-Rhizobium
  - Atopobium
  - Bergeyella
  - Bifidobacterium
  - Blautia
  - Chryseobacterium
  - Empedobacter
  - Enhydrobacter
  - Fenollaria
  - Frederiksenia
  - Fusobacterium
  - Gardnerella
  - Geobacillus
  - Jeotgalicoccus
  - Lactococcus
  - Leptotrichia
  - Luteimonas
  - Marinomonas
  - Methylotenera
  - Moraxella
  - Peptoniphilus
  - Peptostreptococcus
  - Perlucidibaca
  - Sediminibacterium
  - Turicella
  - Vibrio

metaG_only (69 taxa):
  - 
  - Acidovorax
  - Afipia
  - Arachnia
  - Blastococcus
  - Campylobacter_B
  - Campylobacter_D
  - Cardiobacterium
  - Centipeda
  - Collinsella
  - Comamonas
  - Cupriavidus
  - Dermabacter
  - Dialister
  - Eikenella
  - Enemella
  - Enterobacter
  - F0422
  - Faecalibacterium
  - Granulicatella
  - Haematobacter
  - Haemophilus_A
  - Haemophilus_D
  - Hydrogenophaga
  - Hylemonella
  - Kingella_A
  - Kingella_B
  - Kno

### Four-way UpSet diagram with 16S V1-V3, 16S V4, metaG rs210 AND metaG wolr2

In [886]:
# # Load wolr2 table
# metaG_wolr2_path = '../Data/metaG/Tables/metaG_wolr2_micov-filt_genus.biom'
# metaG_wolr2_df = biom_to_df(metaG_wolr2_path)
# metaG_wolr2_df.index = metaG_wolr2_df.index.str.replace('^ g__', '', regex=True)

# # Load rs210 table
# metaG_rs210_path = '../Data/metaG/Tables/metaG_rs210_micov-filt_genus.biom'
# metaG_rs210_df = biom_to_df(metaG_rs210_path)
# metaG_rs210_df.index = metaG_rs210_df.index.str.replace('^ g__', '', regex=True)

# # Step 1: Convert to sets
# filtered_V1V3 = filter_by_prevalence(V1V3_biom, V1V3_biom.index, 0.1)
# filtered_V4 = filter_by_prevalence(V4_biom, V4_biom.index, 0.1)
# filtered_rs210 = filter_by_prevalence(metaG_rs210_df, metaG_rs210_df.index, 0.1)
# filtered_wolr2 = filter_by_prevalence(metaG_wolr2_df, metaG_wolr2_df.index, 0.1)

# # Step 2: Combine all unique taxa across all sources
# all_taxa = sorted(filtered_V1V3 | filtered_V4 | filtered_rs210 | filtered_wolr2)

# # Step 3: Create a binary presence matrix
# presence_matrix = pd.DataFrame(index=all_taxa)
# presence_matrix['V1V3'] = presence_matrix.index.isin(filtered_V1V3)
# presence_matrix['V4'] = presence_matrix.index.isin(filtered_V4)
# presence_matrix['metaG_rs210'] = presence_matrix.index.isin(filtered_rs210)
# presence_matrix['metaG_wolr2'] = presence_matrix.index.isin(filtered_wolr2)

# # Step 4: UpSet plot
# memberships = from_memberships(
#     presence_matrix.apply(lambda row: [col for col in presence_matrix.columns if row[col]], axis=1),
#     data=presence_matrix.index
# )

# plt.figure(figsize=(12, 6))
# upset = UpSet(memberships, show_counts=True, sort_by='cardinality', intersection_plot_elements=15)
# upset.plot()
# plt.suptitle("Shared and Unique Genera Across All 16S and metaG Datasets", fontsize=16)
# plt.tight_layout()
# plt.savefig('../Figures/16S_Figures/primer_comparison/venn_upset_4set.png', dpi=600)
# plt.show()


### Comparison of 16S V1-V3, 16S V4, and shotgun of Cutibacterium per-sample ranked relative abundances with intersectional samples

In [887]:
def get_taxon_ranked_df(taxon_name, v1v3_df, v4_df, shotgun_df):
    # Format the taxon name for each source
    v1v3_taxon = f'{taxon_name}'
    v4_taxon = f'{taxon_name}'
    shotgun_taxon = f'{taxon_name}'
    
    if v1v3_taxon not in v1v3_df.index or v4_taxon not in v4_df.index or not shotgun_taxon:
        raise ValueError(f"Taxon '{taxon_name}' not found in one or more datasets.")
    
    # Extract and transpose
    v1v3_data = v1v3_df.loc[[v1v3_taxon]].T
    v1v3_data.columns = [f'{taxon_name} V1-V3']
    v1v3_data['V1-V3'] = v1v3_data.iloc[:, 0].rank(method='min').astype(int)

    v4_data = v4_df.loc[[v4_taxon]].T
    v4_data.columns = [f'{taxon_name} V4']
    v4_data['V4'] = v4_data.iloc[:, 0].rank(method='min').astype(int)

    shot_data = shotgun_df[[shotgun_taxon]].rename(columns={shotgun_taxon: f'{taxon_name} Shotgun'})
    shot_data['Shotgun'] = shot_data.iloc[:, 0].rank(method='min').astype(int)

    # Standardize index names across datasets
    shot_data.index = shot_data.index.str.replace('_', '.', regex=False)
    v1v3_data.index = v1v3_data.index.astype(str)
    v4_data.index = v4_data.index.astype(str)
    
    # Align sample names
    shared_samples = v1v3_data.index.intersection(v4_data.index).intersection(shot_data.index)
    
    # Merge all
    merged = pd.concat([
        v1v3_data.loc[shared_samples]['V1-V3'],
        v4_data.loc[shared_samples]['V4'],
        shot_data.loc[shared_samples]['Shotgun']
    ], axis=1)
    
    return merged


In [888]:
# Load tables
# shotgun_tbl = pd.read_csv('../Data/metaG/VEBA_analysis/output_files/X_mags.with_taxonomy.with_slcs_name.csv')
# shotgun_tbl_subset = shotgun_tbl.iloc[:, 4:]
# shotgun_tbl_subset = shotgun_tbl_subset.set_index('name')
# shotgun_tbl_subset.index.name = None

# shotgun_tbl



In [889]:
shotgun_tbl = biom_to_df('../Data/metaG/Tables/metaG_wolr2_micov-filt_genus.biom')
shotgun_tbl_subset = shotgun_tbl.iloc[:, 4:]
# shotgun_tbl_subset = shotgun_tbl_subset.set_index('name')
shotgun_tbl_subset.index.name = None

shotgun_tbl

,LAMI.RD306.D14.C1,LAMI.RD306.D0.C3,LAMI.RD310.D16.C2,LAMI.RD310.D0.C2,LAMI.RD310.D14.C3,LAMI.RD310.D7.C3,LAMI.RD310.D0.C3,LAMI.RD310.D21.C2,LAMI.RD304.D11.C1,LAMI.RD304.D0.C1,...,LAMI.RD308.D0.C2,LAMI.RD308.D0.C3,LAMI.RD308.D23.C2,LAMI.RD308.D14.C3,LAMI.RD308.D11.C2,LAMI.RD308.D9.C3,LAMI.RD306.D11.C1,LAMI.RD306.D23.C1,LAMI.RD306.D14.C3,LAMI.RD306.D28.C3
g__Cutibacterium,0.885295,0.957772,0.984448,0.978453,0.994953,0.995983,0.989133,0.980080,0.969096,0.964177,...,0.984800,0.994262,0.874065,0.994297,0.985900,0.969424,0.910988,0.984765,0.936587,0.924127
g__Corynebacterium,0.026314,0.007272,0.003572,0.009076,0.002600,0.001534,0.006265,0.010985,0.004157,0.008947,...,0.002225,0.001651,0.011863,0.001780,0.003103,0.003935,0.014217,0.007951,0.015095,0.020423
g__Actinomyces,0.026079,0.007483,0.000398,0.000726,0.000117,0.000164,0.000445,0.002529,0.003736,0.001358,...,0.002330,0.000913,0.015645,0.000831,0.001429,0.008502,0.010645,0.001745,0.010434,0.014310
g__Micrococcus,0.005293,0.005844,0.001077,0.006148,0.000281,0.000176,0.001604,0.000972,0.003197,0.005855,...,0.001639,0.001230,0.002857,0.000773,0.002541,0.004836,0.006874,0.000539,0.003068,0.010879
g__Staphylococcus,0.005644,0.002459,0.004836,0.000574,0.001066,0.001370,0.000386,0.001335,0.004333,0.006979,...,0.000457,0.000504,0.000539,0.000082,0.000480,0.000504,0.004239,0.001733,0.005890,0.005808
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
g__Mogibacterium,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000082,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
g__Nanogingivalis,0.000047,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000023,0.000000,0.000000,0.000000,0.000082,0.000000,0.000000,0.000000
g__Scardovia,0.000012,0.000000,0.000023,0.000000,0.000000,0.000000,0.000000,0.000000,0.000012,0.000000,...,0.000000,0.000000,0.000082,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
g__Latilactobacillus,0.000035,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000012,0.000000,0.000000,0.000000,0.000047,0.000000,0.000035,0.000000


In [890]:
shotgun_tbl_subset_collapsed = shotgun_tbl_subset.groupby(shotgun_tbl_subset.index).sum()
shotgun_tbl_subset_collapsed = shotgun_tbl_subset_collapsed.T
shotgun_tbl_subset_collapsed

# Save as the VEBA MAGs collapsed Genus-level
biom_table = Table(
    data=shotgun_tbl_subset_collapsed.values,
    observation_ids=shotgun_tbl_subset_collapsed.index.tolist(),
    sample_ids=shotgun_tbl_subset_collapsed.columns.tolist()
)

# Save to file
output_fp = "../Data/metaG/Tables/VEBA_MAGs_collapsed_genus_absolute.biom"
with open(output_fp, "w") as f:
    biom_table.to_json("Generated from pandas DataFrame", f)

In [891]:
relative_abundance_df = shotgun_tbl_subset_collapsed.div(shotgun_tbl_subset_collapsed.sum(axis=0), axis=1)
relative_abundance_df = relative_abundance_df

In [892]:
# Fix sample names
relative_abundance_df.index = (
    relative_abundance_df.index
    .str.replace('_', '.', regex=False)
    .str.split('.')
    .str[:4]
    .str.join('.')
)
relative_abundance_df

,g__,g__Abiotrophia,g__Acidovorax,g__Acinetobacter,g__Actinomyces,g__Aerococcus,g__Afipia,g__Aggregatibacter,g__Alloiococcus,g__Alloprevotella,...,g__Stomatobaculum,g__Streptococcus,g__Tannerella,g__Tepidiphilus,g__Tsuneonella,g__Varibaculum,g__Veillonella,g__Winkia,g__Xanthomonas,g__Yonghaparkia
LAMI.RD310.D14.C3,0.000000,0.000000,0.000000,0.000000,0.001381,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.003161,0.000000,0.00,0.000000,0.000000,0.002262,0.000000,0.000000,0.037037
LAMI.RD310.D7.C3,0.000000,0.000000,0.000000,0.000000,0.001934,0.000000,0.035714,0.000000,0.000000,0.000000,...,0.000000,0.001976,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD310.D0.C3,0.000000,0.000000,0.000000,0.007843,0.005249,0.052632,0.107143,0.000000,0.000000,0.000000,...,0.000000,0.004741,0.000000,0.65,0.000000,0.000000,0.002262,0.000000,0.023669,0.000000
LAMI.RD310.D21.C2,0.066667,0.000000,0.215686,0.013725,0.029834,0.000000,0.107143,0.000000,0.000000,0.000000,...,0.000000,0.006717,0.000000,0.00,0.043478,0.000000,0.002262,0.000000,0.032544,0.074074
LAMI.RD304.D11.C1,0.000000,0.023438,0.078431,0.058824,0.044061,0.052632,0.071429,0.026667,0.133333,0.015411,...,0.000000,0.062426,0.041667,0.00,0.000000,0.056604,0.079186,0.100000,0.005917,0.148148
LAMI.RD304.D0.C1,0.000000,0.007812,0.137255,0.396078,0.016022,0.315789,0.142857,0.013333,0.000000,0.000000,...,0.076923,0.010273,0.000000,0.05,0.086957,0.301887,0.011312,0.066667,0.005917,0.074074
LAMI.RD304.D14.C3,0.000000,0.054688,0.000000,0.015686,0.080249,0.105263,0.000000,0.133333,0.000000,0.015411,...,0.000000,0.166733,0.194444,0.00,0.000000,0.018868,0.122172,0.133333,0.002959,0.000000
LAMI.RD304.D0.C3,0.000000,0.000000,0.019608,0.054902,0.007182,0.157895,0.000000,0.000000,0.000000,0.000000,...,0.076923,0.007112,0.000000,0.25,0.086957,0.037736,0.004525,0.166667,0.000000,0.074074
LAMI.RD304.D7.C1,0.000000,0.015625,0.058824,0.031373,0.019199,0.105263,0.107143,0.000000,0.066667,0.000000,...,0.000000,0.007112,0.000000,0.05,0.043478,0.188679,0.002262,0.233333,0.011834,0.074074
LAMI.RD304.D28.C3,0.000000,0.000000,0.039216,0.000000,0.007182,0.052632,0.035714,0.000000,0.000000,0.000428,...,0.000000,0.004346,0.000000,0.00,0.000000,0.169811,0.004525,0.100000,0.000000,0.000000


In [893]:
# Make a copy to avoid modifying the original
wgs_biom = relative_abundance_df.copy()
wgs_biom.columns = wgs_biom.columns.str.replace('^ g__', '', regex=True)

# Collapse Cutibacterium acnes and granulosum into one column
wgs_biom[' g__Cutibacterium'] = (
    wgs_biom.get('Cutibacterium acnes', 0) + 
    wgs_biom.get('Cutibacterium granulosum', 0)
)

# Rename other specific columns
# wgs_biom = wgs_biom.rename(columns={
#     'Corynebacterium spp.': 'Corynebacterium',
#     'Lawsonella cleavelandensis': 'Lawsonella',
#     'Micrococcus spp.': 'Micrococcus',
#     'Streptococcus mitis': 'Streptococcus'
# })

# Drop the old individual columns that were collapsed or renamed
# wgs_biom = wgs_biom.drop(columns=[
#     'Cutibacterium acnes',
#     'Cutibacterium granulosum'
# ])

# (Optional) View the updated column names
print(wgs_biom.columns.tolist())


['', 'Abiotrophia', 'Acidovorax', 'Acinetobacter', 'Actinomyces', 'Aerococcus', 'Afipia', 'Aggregatibacter', 'Alloiococcus', 'Alloprevotella', 'Arachnia', 'Blastococcus', 'Brachybacterium', 'Bradyrhizobium', 'Brevibacterium', 'Brevundimonas', 'Brochothrix', 'Campylobacter_B', 'Campylobacter_D', 'Capnocytophaga', 'Cardiobacterium', 'Caulobacter', 'Centipeda', 'Collinsella', 'Comamonas', 'Corynebacterium', 'Cupriavidus', 'Cutibacterium', 'Dermabacter', 'Dialister', 'Dolosigranulum', 'Eikenella', 'Enemella', 'Enterobacter', 'F0422', 'Faecalibacterium', 'Finegoldia', 'Gemella', 'Granulicatella', 'Haematobacter', 'Haemophilus', 'Haemophilus_A', 'Haemophilus_D', 'Hydrogenophaga', 'Hylemonella', 'Janibacter', 'Kingella_A', 'Kingella_B', 'Knoellia', 'Kocuria', 'Kytococcus', 'Lactobacillus', 'Lancefieldella', 'Latilactobacillus', 'Lautropia', 'Lawsonella', 'Leclercia', 'Leuconostoc', 'Levilactobacillus', 'Limnobacter', 'Marihabitans', 'Marinilactibacillus', 'Marmoricola', 'Massilia', 'Microbact

In [894]:
# Get shared sample IDs across all three datasets
shared_samples = (
    V1V3_biom.columns
    .intersection(V4_biom.columns)
    .intersection(wgs_biom.index)
)

# Confirm sample count
print(f"Number of matched samples: {len(shared_samples)}")

# Subset each dataset
V1V3_biom = V1V3_biom[shared_samples]
V4_biom = V4_biom[shared_samples]
wgs_biom = wgs_biom.loc[shared_samples]

Number of matched samples: 19


In [895]:
wgs_biom

,,Abiotrophia,Acidovorax,Acinetobacter,Actinomyces,Aerococcus,Afipia,Aggregatibacter,Alloiococcus,Alloprevotella,...,Streptococcus,Tannerella,Tepidiphilus,Tsuneonella,Varibaculum,Veillonella,Winkia,Xanthomonas,Yonghaparkia,g__Cutibacterium
LAMI.RD304.D14.C3,0.000000,0.054688,0.000000,0.015686,0.080249,0.105263,0.000000,0.133333,0.000000,0.015411,...,0.166733,0.194444,0.00,0.000000,0.018868,0.122172,0.133333,0.002959,0.000000,0
LAMI.RD304.D7.C1,0.000000,0.015625,0.058824,0.031373,0.019199,0.105263,0.107143,0.000000,0.066667,0.000000,...,0.007112,0.000000,0.05,0.043478,0.188679,0.002262,0.233333,0.011834,0.074074,0
LAMI.RD304.D0.C1,0.000000,0.007812,0.137255,0.396078,0.016022,0.315789,0.142857,0.013333,0.000000,0.000000,...,0.010273,0.000000,0.05,0.086957,0.301887,0.011312,0.066667,0.005917,0.074074,0
LAMI.RD306.D11.C1,0.100000,0.351562,0.078431,0.090196,0.125552,0.000000,0.142857,0.026667,0.400000,0.017551,...,0.088107,0.013889,0.00,0.086957,0.037736,0.097285,0.000000,0.020710,0.111111,0
LAMI.RD308.D0.C2,0.000000,0.000000,0.000000,0.001961,0.027486,0.000000,0.000000,0.026667,0.000000,0.013271,...,0.028052,0.069444,0.00,0.000000,0.000000,0.013575,0.000000,0.810651,0.000000,0
LAMI.RD304.D0.C3,0.000000,0.000000,0.019608,0.054902,0.007182,0.157895,0.000000,0.000000,0.000000,0.000000,...,0.007112,0.000000,0.25,0.086957,0.037736,0.004525,0.166667,0.000000,0.074074,0
LAMI.RD308.D0.C3,0.000000,0.000000,0.000000,0.003922,0.010773,0.000000,0.000000,0.000000,0.000000,0.000428,...,0.003951,0.000000,0.00,0.000000,0.018868,0.002262,0.033333,0.002959,0.000000,0
LAMI.RD306.D23.C1,0.000000,0.023438,0.039216,0.000000,0.020580,0.000000,0.035714,0.000000,0.000000,0.002568,...,0.005136,0.000000,0.00,0.000000,0.000000,0.020362,0.000000,0.008876,0.111111,0
LAMI.RD306.D14.C3,0.066667,0.140625,0.058824,0.060784,0.123066,0.000000,0.142857,0.013333,0.333333,0.010274,...,0.080601,0.013889,0.00,0.130435,0.018868,0.031674,0.000000,0.002959,0.111111,0
LAMI.RD304.D11.C1,0.000000,0.023438,0.078431,0.058824,0.044061,0.052632,0.071429,0.026667,0.133333,0.015411,...,0.062426,0.041667,0.00,0.000000,0.056604,0.079186,0.100000,0.005917,0.148148,0


In [896]:
taxa_info = {
    "Lawsonella": {"color": "#3030b7"},
    "Cutibacterium": {"color": "#ff7f00"},
    "Streptococcus": {"color": "#c8a165"},
    "Micrococcus": {"color": "#f47fb8"},
    "Corynebacterium": {"color": "#fff44f"},
    "Staphylococcus": {"color": "#92f0f0"},
    "Lactobacillus": {"color": "#daead3"},
    "Alloprevotella": {"color": "#8c5079"},
    "Veillonella": {"color": "#c5bce0"},
    "Rothia": {"color": "#4c72b0"}
    
}

In [897]:
wgs_biom

,,Abiotrophia,Acidovorax,Acinetobacter,Actinomyces,Aerococcus,Afipia,Aggregatibacter,Alloiococcus,Alloprevotella,...,Streptococcus,Tannerella,Tepidiphilus,Tsuneonella,Varibaculum,Veillonella,Winkia,Xanthomonas,Yonghaparkia,g__Cutibacterium
LAMI.RD304.D14.C3,0.000000,0.054688,0.000000,0.015686,0.080249,0.105263,0.000000,0.133333,0.000000,0.015411,...,0.166733,0.194444,0.00,0.000000,0.018868,0.122172,0.133333,0.002959,0.000000,0
LAMI.RD304.D7.C1,0.000000,0.015625,0.058824,0.031373,0.019199,0.105263,0.107143,0.000000,0.066667,0.000000,...,0.007112,0.000000,0.05,0.043478,0.188679,0.002262,0.233333,0.011834,0.074074,0
LAMI.RD304.D0.C1,0.000000,0.007812,0.137255,0.396078,0.016022,0.315789,0.142857,0.013333,0.000000,0.000000,...,0.010273,0.000000,0.05,0.086957,0.301887,0.011312,0.066667,0.005917,0.074074,0
LAMI.RD306.D11.C1,0.100000,0.351562,0.078431,0.090196,0.125552,0.000000,0.142857,0.026667,0.400000,0.017551,...,0.088107,0.013889,0.00,0.086957,0.037736,0.097285,0.000000,0.020710,0.111111,0
LAMI.RD308.D0.C2,0.000000,0.000000,0.000000,0.001961,0.027486,0.000000,0.000000,0.026667,0.000000,0.013271,...,0.028052,0.069444,0.00,0.000000,0.000000,0.013575,0.000000,0.810651,0.000000,0
LAMI.RD304.D0.C3,0.000000,0.000000,0.019608,0.054902,0.007182,0.157895,0.000000,0.000000,0.000000,0.000000,...,0.007112,0.000000,0.25,0.086957,0.037736,0.004525,0.166667,0.000000,0.074074,0
LAMI.RD308.D0.C3,0.000000,0.000000,0.000000,0.003922,0.010773,0.000000,0.000000,0.000000,0.000000,0.000428,...,0.003951,0.000000,0.00,0.000000,0.018868,0.002262,0.033333,0.002959,0.000000,0
LAMI.RD306.D23.C1,0.000000,0.023438,0.039216,0.000000,0.020580,0.000000,0.035714,0.000000,0.000000,0.002568,...,0.005136,0.000000,0.00,0.000000,0.000000,0.020362,0.000000,0.008876,0.111111,0
LAMI.RD306.D14.C3,0.066667,0.140625,0.058824,0.060784,0.123066,0.000000,0.142857,0.013333,0.333333,0.010274,...,0.080601,0.013889,0.00,0.130435,0.018868,0.031674,0.000000,0.002959,0.111111,0
LAMI.RD304.D11.C1,0.000000,0.023438,0.078431,0.058824,0.044061,0.052632,0.071429,0.026667,0.133333,0.015411,...,0.062426,0.041667,0.00,0.000000,0.056604,0.079186,0.100000,0.005917,0.148148,0


In [898]:
fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(111, projection='3d')

legend_items = []

for taxon, style in taxa_info.items():
    try:
        # Get sample-wise ranked data for this taxon
        df = get_taxon_ranked_df(
            taxon_name=taxon,
            v1v3_df=V1V3_biom,
            v4_df=V4_biom,
            shotgun_df=wgs_biom
        )

        x, y, z = df['V1-V3'].values, df['V4'].values, df['Shotgun'].values

        # PCA-style best-fit line in 3D
        points = np.stack([x, y, z], axis=1)
        mean_point = points.mean(axis=0)
        centered_points = points - mean_point
        pca_line = PCA(n_components=1)
        pca_line.fit(centered_points)
        direction = pca_line.components_[0]

        # Create and plot PCA line
        t = np.linspace(-20, 20, 100)
        line = mean_point + np.outer(t, direction)
        ax.plot(line[:, 0], line[:, 1], line[:, 2],
                color=style['color'], linewidth=2, alpha=0.8)

        # Compute multivariate R²
        pca_full = PCA(n_components=3)
        pca_full.fit(centered_points)
        r_squared = pca_full.explained_variance_ratio_[0]

        # Plot scatter points and collect legend item
        scatter = ax.scatter(x, y, z, color=style['color'],
                             edgecolors='k', s=50, alpha=0.7)
        legend_label = f"{taxon} (R² = {r_squared:.2f})"
        legend_items.append((scatter, legend_label, r_squared))

    except Exception as e:
        print(f"Skipped {taxon}: {e}")

# Sort legend items by R² descending
legend_items_sorted = sorted(legend_items, key=lambda x: x[2], reverse=True)
handles, labels = zip(*[(item[0], item[1]) for item in legend_items_sorted])

# Plot styling
ax.set_xlabel('16S V1–V3 Rank', fontsize=16)
ax.set_ylabel('16S V4 Rank', fontsize=16)
ax.set_zlabel('metaG Rank', fontsize=16)
ax.set_title('Cross-platform Correlation per Taxon', fontsize=20)

# Set tick label font sizes
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.tick_params(axis='z', labelsize=12)

# Remove negative axis values
ax.set_xlim(0, ax.get_xlim()[1])
ax.set_ylim(0, ax.get_ylim()[1])
ax.set_zlim(0, ax.get_zlim()[1])

# Add legend
ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(-0.1, 1), fontsize=12)

# Final output
plt.tight_layout()
plt.savefig('../Figures/multi-omics_Figures/3D-plot_16SV1V3_16SV4_shotgun_top-taxa-corr-R2.png', dpi=600)
plt.show()

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_59393/3412562379.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
